In [1]:
import pandas as pd
import numpy as np
TMI=pd.read_excel('TMI4-import.xlsx')

# qx dan px LK
qx0=np.array(TMI['Laki-Laki'])
px0=1-qx0

# qx dan px PR
qx1=np.array(TMI['Perempuan'])
px1=1-qx1

xt=np.array(TMI['Usia'])

# Fungsi PV
def v(t,i):
  return (1+i)**(-t)

#Fungsi kpx
def p(x,t,jk):
  if t==0:
    px=1
  else:
    if jk==0: # Kalau Laki-Laki
      px=1
      for y in range(x,x+t):
        px=px*(px0[y])
    else: # Kalau Perempuan
      px=1
      for y in range(x,x+t):
        px=px*(px1[y])
  return px
    
# Fungsi PTN Berjangka
def An(i,x,n,jk):
  Ak=0
  for k in range(n):
    Ak=Ak+(v(k+1,i)*p(x,k,jk)*(1-p(x+k,1,jk)))
  return Ak

# Fungsi Anuitas Hidup Berjangka
def an(i,x,n,jk):
    an=0
    for r in range(n):
        an=an+(v(r,i)*p(x,r,jk))
    return an

In [2]:
# TMI Fractional
def fractional(file_tmi, jeka):
    tmi = pd.read_excel(file_tmi)
    rows = []
    for i in range(len(tmi)):
        age = tmi.loc[i, "Usia"]
        if jeka == 0:
            qx = tmi.loc[i, "Laki-Laki"]
        else:
            qx = tmi.loc[i, "Perempuan"]
        for m in range(12):
            s = m/12
            q_month = (qx/12) / (1 - s*qx)
            p_month = 1 - q_month
            rows.append([age, m+1, age+s, q_month, p_month])
    result = pd.DataFrame(rows,columns=["Usia","Bulan","Usia_Fraction","q_bulan","p_bulan"])
    return result
    
# Fungsi PV Bulanan
def v_bulanan(tk, ib):
    return (1 + ib)**(-tk)
# Anuitas Bulanan
def life_annuity_monthly(tmi_fraction, umur, n, ib):
    data = tmi_fraction[tmi_fraction["Usia"] >= umur].reset_index(drop=True)
    surv = data["p_bulan"].cumprod().shift(1).fillna(1)
    annuity = 0
    periode = 12 * n
    for k in range(periode):
        t = k/12
        annuity += v_bulanan(t, ib) * surv.iloc[k]
    annuity = annuity / 12
    return annuity

In [3]:
# Outflow 1 (Manfaat Meninggal Rp 500.000.000 Jangka 59 tahun bunga 6% Laki-Laki)
Outflow1 = An(0.06,40,59,0)*500000000
print(Outflow1)

64558147.33261311


In [4]:
# Anuitas Hidup
Anuitas_Hidup = an(0.06,40,58,0)
print(Anuitas_Hidup)

15.372048495512079


In [5]:
# Outflow 2 (Biaya Alokasi Pure Premium sebesar 40% di tahun pertama dan 0% selanjutnya)
Outflow2 = 0.4
print(Outflow2)

0.4


In [6]:
# Outflow 3 (Biaya Pemeliharaan 5,5% dari pure premium setiap tahun)
Outflow3 = 0.055*an(0.06,40,59,0)
print(Outflow3)

0.8455209856930901


In [7]:
# Outflow 4 (Biaya Administrasi Rp. 30.000 setiap bulan)
tmi_frac = fractional('TMI4-import.xlsx',0)
ax = life_annuity_monthly(tmi_frac,40,59,0.06)
Outflow4 = 360000*ax
print(Outflow4)

5367470.598749933


In [8]:
# Outflow 5 Manfaat Investasi Rp 115.409.560.000
Outflow5 = 115409560000*p(40,59,0)*v(59,0.06)
print(Outflow5)

81679244.26147796


In [26]:
# Gross Premium
Pembilang = Outflow1 + Outflow5 + Outflow4
Penyebut = Anuitas_Hidup - Outflow2 - Outflow3 
GP = Pembilang/Penyebut
print(GP)

10383633.253727553
